In [2]:
import os
import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from alpaca_keys import AlpacaKeys

from alpaca.trading.client import TradingClient
from alpaca.data.historical.stock import StockHistoricalDataClient

from alpaca.data.requests import StockBarsRequest
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, OrderType, TimeInForce

from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

In [5]:
from alpaca.trading.client import TradingClient

api_key = "PKEMVBCQ5QQV74SQUFLL5TQXYO"
secret_key = "Ey1jdpqGebTamfdbk2jqHPEqm4nifCGGH9pacAsESTJn"

trade_client = TradingClient(api_key, secret_key, paper=True)

pairs = [
    ("V", "MA"),
    ("HD", "LOW"),
    ("XOM", "CVX"),
    ("JPM", "BAC"),
    ("GS", "MS"),
    ("UNP", "CSX"),
    ("TGT", "WMT"),
    ("COST", "WMT"),
    ("CVS", "WBA"),
    ("DAL", "UAL"),
]

unique_tickers = sorted(set([ticker for pair in pairs for ticker in pair]))

rows = []

for ticker in unique_tickers:
    try:
        asset = trade_client.get_asset(ticker)

        rows.append({
            "symbol": ticker,
            "name": asset.name,
            "exchange": asset.exchange,
            "status": asset.status,
            "tradable": asset.tradable,
            "shortable": asset.shortable,
            "easy_to_borrow": asset.easy_to_borrow,
            "fractionable": asset.fractionable
        })

    except Exception as e:
        rows.append({
            "symbol": ticker,
            "name": None,
            "exchange": None,
            "status": None,
            "tradable": False,
            "shortable": False,
            "easy_to_borrow": False,
            "fractionable": False,
            "error": str(e)
        })

assets_df = pd.DataFrame(rows)
assets_df

,symbol,name,exchange,status,tradable,shortable,easy_to_borrow,fractionable
0,BAC,Bank of America Corporation,AssetExchange.NYSE,AssetStatus.ACTIVE,True,True,True,True
1,COST,Costco Wholesale Corporation Common Stock,AssetExchange.NASDAQ,AssetStatus.ACTIVE,True,True,True,True
2,CSX,CSX Corporation Common Stock,AssetExchange.NASDAQ,AssetStatus.ACTIVE,True,True,True,True
3,CVS,CVS HEALTH CORPORATION,AssetExchange.NYSE,AssetStatus.ACTIVE,True,True,True,True
4,CVX,Chevron Corporation,AssetExchange.NYSE,AssetStatus.ACTIVE,True,True,True,True
5,DAL,"Delta Air Lines, Inc.",AssetExchange.NYSE,AssetStatus.ACTIVE,True,True,True,True
6,GS,Goldman Sachs Group Inc.,AssetExchange.NYSE,AssetStatus.ACTIVE,True,True,True,True
7,HD,"Home Depot, Inc.",AssetExchange.NYSE,AssetStatus.ACTIVE,True,True,True,True
8,JPM,JPMorgan Chase & Co.,AssetExchange.NYSE,AssetStatus.ACTIVE,True,True,True,True
9,LOW,Lowe's Companies Inc.,AssetExchange.NYSE,AssetStatus.ACTIVE,True,True,True,True


In [4]:
pair_rows = []

for a, b in pairs:
    a_info = assets_df[assets_df["symbol"] == a].iloc[0]
    b_info = assets_df[assets_df["symbol"] == b].iloc[0]

    pair_rows.append({
        "pair": f"{a}/{b}",
        f"{a}_tradable": a_info["tradable"],
        f"{b}_tradable": b_info["tradable"],
        f"{a}_shortable": a_info["shortable"],
        f"{b}_shortable": b_info["shortable"],
        f"{a}_easy_to_borrow": a_info["easy_to_borrow"],
        f"{b}_easy_to_borrow": b_info["easy_to_borrow"],
        f"{a}_fractionable": a_info["fractionable"],
        f"{b}_fractionable": b_info["fractionable"],
        "pair_tradeable_for_long_short": (
            a_info["tradable"] and b_info["tradable"] and
            a_info["shortable"] and b_info["shortable"]
        )
    })

pair_check_df = pd.DataFrame(pair_rows)
pair_check_df

,pair,V_tradable,MA_tradable,V_shortable,MA_shortable,V_easy_to_borrow,MA_easy_to_borrow,V_fractionable,MA_fractionable,pair_tradeable_for_long_short,...,CVS_fractionable,WBA_fractionable,DAL_tradable,UAL_tradable,DAL_shortable,UAL_shortable,DAL_easy_to_borrow,UAL_easy_to_borrow,DAL_fractionable,UAL_fractionable
0,V/MA,True,True,True,True,True,True,True,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HD/LOW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,XOM/CVX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JPM/BAC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GS/MS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,UNP/CSX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,TGT/WMT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,COST/WMT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,CVS/WBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,...,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,DAL/UAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,...,NaN,NaN,True,True,True,True,True,True,True,True
